# Assess Micro-Scale Inverse modelling Processes (IMP)

Using high resolution simulation to generate flux maps for low resolution simulations. The low-res flux maps are used during parameter estimation. This method allows to neglect the heater geometry.




In [ ]:
import os
import re
import sys
import fdsreader
import matplotlib

import numpy as np
import scipy as sp
import pandas as pd
import firescipy as fsp
import utilities as utils

import matplotlib.pyplot as plt

from importlib import reload  # Python 3.4+
from scipy.ndimage import uniform_filter1d
from scipy.signal import savgol_filter

# fsp_dev_path = os.path.join("..", "..", "..", "Kinetics", "ReactionRateDev", "GeneralInformation")
# # Add the directory containing your module to the Python path (wants absolute paths)
# sys.path.append(os.path.abspath(fsp_dev_path))
# print(os.path.abspath(fsp_dev_path))
# # sys.path.append(fsp_dev_path)
# import FireSciPy as fsp
# reload(fsp)

reload(utils)


In [ ]:
###########################################################################
## ! Use the 'requirements.txt' to create a virtual Python environment ! ##
###########################################################################

# Package Versions
# ----------------
# Python version: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
# Python path: F:\PhD\BESKID\PyrolysisModelling\GeneralInformation\venv_BESKID_Pyro\Scripts\python.exe
# Numpy version: 2.3.2
# SciPy version: 1.16.1
# Pandas version: 2.3.1
# FireSciPy version: 0.0.5  # py -m pip install --index-url https://test.pypi.org/simple/ --no-deps firescipy==0.0.5
# Matplotlib version: 3.10.5
# RegEx version: 2.2.1


print('Package Versions')
print('----------------')
print('Python version: {}'.format(sys.version))
print('Python path: {}'.format(sys.executable))
print('Numpy version: {}'.format(np.__version__))
print('SciPy version: {}'.format(sp.__version__))
print('Pandas version: {}'.format(pd.__version__))
print('fdsreader version: {}'.format(fdsreader.__version__))
print('FireSciPy version: {}'.format(fsp.__version__))
print('Matplotlib version: {}'.format(matplotlib.__version__))
print('RegEx version: {}'.format(re.__version__))


In [ ]:
# General information.

tga_tupo_root = os.path.join("..", "Experiments", "BESKID_ExperimentalData", "Materials", 
                             "PMMA", "DE_6mm", "TGA")


hr_labels_tupo = ['5Kmin', '10Kmin', '20Kmin', '30Kmin', '50Kmin']
nominal_masses_tupo = ['8.6mg']



# Order of plot colors
plot_colors = ["tab:blue", "tab:orange", "tab:green", 
               "tab:red", "tab:purple", "tab:brown", 
               "tab:pink", "tab:gray", "tab:olive", 
               "tab:cyan"]

# Basic plot settings
plt.rcParams.update({
    'axes.axisbelow': True,     # Keep grid behind plots
    'figure.autolayout': True,  # Equivalent to calling tight_layout()
    'axes.facecolor': 'white',  # Prevents transparent background
    'grid.alpha': 0.6,          # Makes gridlines more readable
    'font.size': 12             # Set global font size
})


# Directory used to collect the output produced by this notebook.
output_dir = "PrepareMicroScaleExperiments_Output"
if not os.path.isdir(output_dir):
    os.mkdir(output_dir)
    print("* Output directory created.")
else:
    print("* Output directory already exists.")


In [ ]:

# def get_window_points(phi, Δ_phi):
    
#     # Get resolution
#     d_phi = np.median(np.diff(phi))
    
#     # Determine window size for smoothing
#     win_pts = int(round(Δ_phi / d_phi))
#     win_pts = max(win_pts, 5)
    
#     # Ensure odd number window size
#     if win_pts % 2 == 0:
#         win_pts += 1
        
#     return win_pts



# def H_baseline(m_0, dT_dt, c_T, m_t):
#     """
#     Compute sensible heat flow baseline, to later determine the specific heat capacities.

#     From: Formula 11 in 'Measurement of kinetics and thermodynamics of the
#     thermal degradation for non-charring polymers',
#     Jing Li, Stanislav I.Stoliarov, Combustion and Flame 160 (2013) 1287–1297
#     https://doi.org/10.1016/j.combustflame.2013.02.012
#     https://doi.org/10.1016/j.polymdegradstab.2013.09.022

#     :param m_0: initial sample mass [mg]
#     :param dT_dt: instantaneous heating rate [K/s]
#     :param c_T: list of c_j [J / (kg K)]
#     :param m_t: list of m_j [mg]
#     :return: baseline as numpy.ndarray
#     """

#     c_m = list()

#     for i, c_i_T in enumerate(c_T):
#         c_m_i = c_i_T * m_t[i]
#         c_m.append(c_m_i)

#     if len(c_m) > 1:
#         sum_c_m = np.sum(c_m, axis=0)
#     else:
#         sum_c_m = c_m[0]

#     baseline = 1/m_0 * dT_dt * sum_c_m

#     return baseline



# Thermogravimetric Analysis (TGA)

## Read Experimental Data

### TUPO

In [ ]:

# Where does the data start?
tupo_header_lines = {
    "ExpDat_PMMA_DE_6mm_10Kmin_01.txt": 36,
    "ExpDat_PMMA_DE_6mm_10Kmin_02.txt": 36,
    "ExpDat_PMMA_DE_6mm_10Kmin_03.txt": 36,
    "ExpDat_PMMA_DE_6mm_20Kmin_01.txt": 35,
    "ExpDat_PMMA_DE_6mm_20Kmin_02.txt": 35,
    "ExpDat_PMMA_DE_6mm_20Kmin_03.txt": 35,
    "ExpDat_PMMA_DE_6mm_30Kmin_01.txt": 33,
    "ExpDat_PMMA_DE_6mm_30Kmin_02.txt": 33,
    "ExpDat_PMMA_DE_6mm_30Kmin_03.txt": 33,
    "ExpDat_PMMA_DE_6mm_30Kmin_04.txt": 33,
    "ExpDat_PMMA_DE_6mm_30Kmin_05.txt": 33,
    "ExpDat_PMMA_DE_6mm_30Kmin_06.txt": 33,
    "ExpDat_PMMA_DE_6mm_50Kmin_01.txt": 35,
    "ExpDat_PMMA_DE_6mm_50Kmin_02.txt": 35,
    "ExpDat_PMMA_DE_6mm_50Kmin_03.txt": 35,
    "ExpDat_PMMA_DE_6mm_50Kmin_04.txt": 35,
    "ExpDat_PMMA_DE_6mm_50Kmin_05.txt": 35,
    "ExpDat_PMMA_DE_6mm_50Kmin_06.txt": 35,
    "ExpDat_PMMA_DE_6mm_5Kmin_01.txt": 35,
    "ExpDat_PMMA_DE_6mm_5Kmin_02.txt": 35,
    "ExpDat_PMMA_DE_6mm_5Kmin_03.txt": 35 
}


In [ ]:
# Read meta data
tupo_meta = dict()


for element in os.listdir(tga_tupo_root):
    if "ExpDat_" in element and "_6mm_" in element:
#         print(element)
        
        label_parts = element[:-4].split('_')
        thickness = label_parts[3]
        heating_rate = label_parts[4]
        repetition = label_parts[5]
        
        rep_label = f"Rep_{repetition}"
        
        keys = [thickness, heating_rate, rep_label]
        fsp.utils.ensure_nested_dict(
            nested_dict=tupo_meta, 
            keys=keys)

        meta_lines = dict()
        exp_path = os.path.join(tga_tupo_root, element)
        
        first_n = tupo_header_lines[element]
        with open(exp_path, "r") as file:
            for line in range(first_n):
                line = next(file).strip()
                line_parts = line.split(';')
                key = line_parts[0].strip()[1:-1]
                value = line_parts[1]
                
                meta_lines[key] = value
                
        fsp.utils.store_in_nested_dict(
            nested_dict=tupo_meta, 
            new_data=meta_lines, 
            keys=keys)


# Check result
tupo_meta["6mm"]["10Kmin"]["Rep_01"]
    

In [ ]:

# Check which segments cover the heat-up of the sample
segment_label = 'SEG. 2'  # segment 2

for hr_label in tupo_meta["6mm"]:
    
    hr_data = tupo_meta["6mm"][hr_label]
    for rep_label in hr_data:
        seg_2 = hr_data[rep_label][segment_label]
        print(seg_2)


In [ ]:
# Read experimental data
tupo_exp = dict()

# Which segment to read
segment = 2


for element in os.listdir(tga_tupo_root):
    if "ExpDat_" in element and "_6mm_" in element:
        print(element)
        
        label_parts = element[:-4].split('_')
        thickness = label_parts[3]
        heating_rate = label_parts[4]
        repetition = label_parts[5]
        
        rep_label = f"Rep_{repetition}"
        
        keys = [thickness, heating_rate, rep_label]
        fsp.utils.ensure_nested_dict(
            nested_dict=tupo_exp, 
            keys=keys)
        
        exp_path = os.path.join(tga_tupo_root, element)
        exp_data = pd.read_csv(
            exp_path, 
            header=tupo_header_lines[element], 
            delimiter=";", 
            encoding='cp1252')
        
        # 2.2) Adjust the first column header
        headers = list(exp_data)
        if '##Temp./ï¿½C' in headers:
            exp_data.rename({'##Temp./ï¿½C': 'Temp./°C'}, axis='columns', inplace=True)
        else:
            exp_data.rename({'##Temp./°C': 'Temp./°C'}, axis='columns', inplace=True)
        
        if 'Segment' in headers:
            short = exp_data.loc[exp_data['Segment'] == 2]
        else:
            print('No segment')
            short = exp_data
        
        fsp.utils.store_in_nested_dict(
            nested_dict=tupo_exp, 
            new_data=short, 
            keys=keys)
    

In [ ]:

tupo_exp["6mm"]["10Kmin"]["Rep_01"].head()


In [ ]:
full = tupo_exp["6mm"]["10Kmin"]["Rep_01"]
short = full.loc[full['Segment'] == 2]

full.tail()


In [ ]:

pmma_6mm = tupo_exp["6mm"]



# Smoothing parameters
Δ_T = 10  # Kelvin
savgol_poly = 3



for hr_label in pmma_6mm:
    
    hr_data = pmma_6mm[hr_label]
    for rep_label in hr_data:
        
        time = hr_data[rep_label]["Time/min"]
        temp = hr_data[rep_label]["Temp./°C"]
        mass = hr_data[rep_label]["Mass/%"]

        # Smooth data
        win_pts = utils.get_window_points(temp, Δ_T)
        mass_smooth = savgol_filter(mass, window_length=win_pts, polyorder=savgol_poly)
    
        
        
        plt.plot(temp, mass_smooth)
        

        
# Plot meta data
plt.xlabel("Sample Temeprature / °C")
plt.ylabel("Normalise Residual Mass / %")

plt.xlim(right=500)

plt.grid()


In [ ]:

pmma_6mm = tupo_exp["6mm"]



# Smoothing parameters
Δ_T = 10  # Kelvin
savgol_poly = 3



for hr_label in pmma_6mm:
    print(hr_label[:-4])
    
    hr_data = pmma_6mm[hr_label]
    for rep_label in hr_data:
        
        time = hr_data[rep_label]["Time/min"]
        temp = hr_data[rep_label]["Temp./°C"]
        mass = hr_data[rep_label]["Mass/%"] / 100

        # Smooth data
        win_pts = utils.get_window_points(temp, Δ_T)
        mass_smooth = savgol_filter(mass, window_length=win_pts, polyorder=savgol_poly)
        
        mlr = -np.gradient(mass, time*60)
        mlr_smooth = -np.gradient(mass_smooth, time*60)
    
        
        
        plt.plot(temp, mlr_smooth)
        

        
# Plot meta data
plt.xlabel("Sample Temeprature / °C")
plt.ylabel("Normalise Mass Loss Rate / 1/s")

plt.xlim(right=500)

plt.grid()


In [ ]:

pmma_6mm = tupo_exp["6mm"]



# Smoothing parameters
Δ_T = 10  # Kelvin
savgol_poly = 3



for hr_label in pmma_6mm:
    
    hr_data = pmma_6mm[hr_label]
    for rep_label in hr_data:
        
        time = hr_data[rep_label]["Time/min"] * 60  # in seconds
        temp = hr_data[rep_label]["Temp./°C"]
        mass = hr_data[rep_label]["Mass/%"] / 100
    
        heating_rate = np.gradient(temp, time/60)  # K/min
        
        plt.plot(time, heating_rate)
        

        
    # Plot meta data
    plt.xlabel("Sample Temeprature / °C")
    plt.ylabel("Normalise Mass Loss Rate / 1/s")

    # plt.xlim(right=500)

    plt.grid()
    plt.show()


In [ ]:


# Initialise data structure for the kinetics assessment
tupo_isoconv = fsp.pyrolysis.kinetics.initialize_investigation_skeleton(
    material=f"PMMA, BESKID",
    investigator="TUPO",
    instrument="TGA/STA",
    date="2025",
    notes="NETZSCH STA 449F3",
    signal={"name": "Mass", "unit": "%"})


# 
pmma_6mm = tupo_exp["6mm"]


# Smoothing parameters
Δ_T = 10  # Kelvin
savgol_poly = 3


for hr_label in pmma_6mm:
    
    hr_data = pmma_6mm[hr_label]
    for rep_label in hr_data:
        
        hr_value = float(hr_label[:-4])
        
        # Get recorded data
        exp_df = hr_data[rep_label]
        
        # Create deep copy of the DataFrame
        n_df = exp_df[['Time/min', 'Temp./°C', 'Mass/%']].copy()
        
        # Preprocess data
        n_df['Time/min'] = n_df['Time/min'].multiply(60)  # in seconds
        n_df['Temp./°C'] = n_df['Temp./°C'] + 273.15  # in Kelvin
        
        # Rename column headers
        n_df.rename(columns={'Time/min': 'Time', 'Temp./°C': 'Temperature', 'Mass/%': 'Mass'}, inplace=True)
        
        # Add DataFrame to database
        fsp.pyrolysis.kinetics.add_constant_heating_rate_tga(
            database=tupo_isoconv,
            condition=hr_label,
            repetition=rep_label,
            raw_data=n_df,
            data_type="integral",
            set_value=[hr_value, "K/min"])


# Adjust column mapping for later functions
# Time (s)  Temperature (K)  Mass (mg)
column_mapping = {
        'time': 'Time',
        'temp': 'Temperature',
        'signal': 'Mass'}

        
# Define conversion fractions where to evaluate the activation energy
# Note: commonly, they range between (0.05 < α < 0.95) or (0.1 < α < 0.9)
# conversion_levels = np.linspace(0.05, 0.95, 37)  # Δα = 2.5
conversion_levels = np.linspace(0.01, 0.99, 99)  # Δα = 1.0
# conversion_levels = np.linspace(0.01, 0.99, 199)  # Δα = 0.5


# Process the experimental data for isoconversional analysis
for hr_label in tupo_isoconv["experiments"]["TGA"]["constant_heating_rate"]:
    # Compute averages and standard deviations per heating rate
    fsp.pyrolysis.kinetics.combine_repetitions(
        database=tupo_isoconv,
        condition=hr_label,
        column_mapping=column_mapping)

# Compute conversion of the experimental data
fsp.pyrolysis.kinetics.compute_conversion(
    database=tupo_isoconv,
    condition="all",
    setup="constant_heating_rate")

# Compute conversion levels for the Ea assessment
fsp.pyrolysis.kinetics.compute_conversion_levels(
    database=tupo_isoconv,
    desired_levels=conversion_levels,
    setup="constant_heating_rate",
    condition="all")

# Compute the activation energy using the KAS method with Starink
fsp.pyrolysis.kinetics.compute_Ea_KAS(
    database=tupo_isoconv,
    B=1.92,
    C=1.0008)
    

In [ ]:

const_hr = tupo_isoconv["experiments"]["TGA"]["constant_heating_rate"]

hr_labels = [
    "5Kmin",
    "10Kmin",
    "20Kmin",
    "30Kmin",
    "50Kmin"
]


for hr_label in hr_labels:
    print(hr_label)
    combined = const_hr[hr_label]["combined"]
    
    temp = combined["Temperature_Avg"] - 273.15
    mass = combined["Mass_Avg"]
    print(np.asarray(mass)[0])
    print(np.asarray(mass)[-1])
    
    plt.plot(temp,
             mass / 100)

    
# Plot meta data
plt.xlabel("Sample Temperature / °C")
plt.ylabel("Normalised Residual Mass / mg/mg")

plt.xlim(left=90, right=510)

plt.grid()



In [ ]:

const_hr = tupo_isoconv["experiments"]["TGA"]["constant_heating_rate"]

hr_labels = [
    "5Kmin",
    "10Kmin",
    "20Kmin",
    "30Kmin",
    "50Kmin"
]


# Smoothing parameters
Δ_T = 10  # Kelvin
savgol_poly = 3


for hr_label in hr_labels:
    print(hr_label)
    combined = const_hr[hr_label]["combined"]
    
    time = combined["Time"]
    temp = combined["Temperature_Avg"]
    mass = combined["Mass_Avg"] / 100    
    
    # Smooth data
    win_pts = utils.get_window_points(temp, Δ_T)
    mass_smooth = savgol_filter(mass, window_length=win_pts, polyorder=savgol_poly)
    
    
    mlr = -np.gradient(mass, time)
    mlr_smooth = -np.gradient(mass_smooth, time)
        
    
    plt.plot(temp,
             mlr_smooth)

    
# Plot meta data
plt.xlabel("Sample Temperature / K")
plt.ylabel("Normalised Mass Loss Rate / 1/s")

plt.xlim(right=800)

plt.grid()



In [ ]:

Ea_results_KAS = tupo_isoconv["experiments"]["TGA"]["Ea_results_KAS"]
Ea = Ea_results_KAS["Ea"]/1000


fig, ax = plt.subplots()

# Plot the Ea against conversion.
conv = Ea_results_KAS["Conversion"]
plt.scatter(conv,
            Ea,
            marker=".", s=42,
            facecolors="none",
            edgecolors=plot_colors[0],
            label=f"KAS Method")




# Shaded areas to indicate first/last 5 % (typically excluded)
x_min = -0.025
x_max = 1.025
ax.axvspan(x_min, 0.05, color='gray', alpha=0.3, label="First / Last 5%")
ax.axvspan(0.95, x_max, color='gray', alpha=0.3)


# Plot meta data.
# ax.set_title(f"Activation Energy (KAS), MaCFP-4 Pine Wood")
ax.set_xlabel("Conversion ($\\alpha$) / -")
ax.set_ylabel("Apparent Activation Energy (E$_a$) / kJ/mol")

ax.set_xlim(left=x_min, right=x_max)
ax.set_ylim(bottom=-7, top=257)
# ax.set_ylim(bottom=200, top=230)

plt.tight_layout()
ax.legend(loc="lower center")
ax.grid()



### Write IMP Target Data (TGA)



In [ ]:

const_hr = tupo_isoconv["experiments"]["TGA"]["constant_heating_rate"]


for hr_label in hr_labels:
    print(hr_label)
    
    combined = const_hr[hr_label]["combined"]
    print(combined["Temperature_Avg"][0]-273.15)
    
    # Write data to file
    file_label = f"TGA_{hr_label}.csv"
    file_path = os.path.join(output_dir, file_label)
    
    combined.to_csv(file_path, index=False)
    

In [ ]:

# Check files

for hr_label in hr_labels:
    file_label = f"TGA_{hr_label}.csv"
    check_path = os.path.join(output_dir, file_label)

    check_df = pd.read_csv(check_path, header=0)


    plt.plot(check_df["Temperature_Avg"]-273.15,
             check_df["Mass_Avg"]*(1/100))

    
# Plot meta data
plt.xlabel("Sample Temperature / °C")
plt.ylabel("Normalised Mass Loss Rate / 1/s")

# plt.xlim(right=500)

plt.grid()


check_df.head()



# Differential Scanning Calorimetry (DSC)

### TUPO

In [ ]:

pmma_6mm = tupo_exp["6mm"]['5Kmin']

pmma_6mm['Rep_01'].head()



### Write IMP Target Data (DSC)



In [ ]:

# Combine DSC data series

hr_label = '5Kmin'
# raw_data = tupo_exp["6mm"][hr_label]

# # Create deep copy of the DataFrame
# n_df = raw_data[['Time/min', 'Temp./°C', 'DSC/(mW/mg)']].copy()

# # Preprocess data
# n_df['Time/min'] = n_df['Time/min'].multiply(60)  # in seconds
# n_df['Temp./°C'] = n_df['Temp./°C'] + 273.15  # in Kelvin




raw_data = dict()

for rep_label in list(tupo_exp["6mm"][hr_label]):
    
    rep_data = tupo_exp["6mm"][hr_label][rep_label]
    
    n_df = rep_data[['Time/min', 'Temp./°C', 'DSC/(mW/mg)']].copy()

    # Preprocess data
    n_df['Time/min'] = n_df['Time/min'].multiply(60)  # in seconds
    n_df['Temp./°C'] = n_df['Temp./°C'] + 273.15  # in Kelvin
    
    
    raw_data[rep_label] = n_df




# Set column mapping from user input
time_col = "Time/min"
temp_col = "Temp./°C"
signal_col = "DSC/(mW/mg)"

# Final column names to be used
time_col_default = "Time"
temp_col_default = "Temperature"
signal_col_default = "Heat_Flow"


# Step 1: Find reference time from longest series
longest_time = None
for rep in raw_data.values():
    if longest_time is None or rep[time_col].iloc[-1] > longest_time.iloc[-1]:
        longest_time = rep[time_col]
reference_time = longest_time.values

# Step 2: Interpolate all repetitions to reference time
combined_data = {time_col_default: reference_time}
for rep_name, rep_data in raw_data.items():
    for col, col_default in [(temp_col, temp_col_default), (signal_col, signal_col_default)]:
        if col not in rep_data.columns:
            raise ValueError(f"Missing column '{col}' in repetition '{rep_name}'.")

        combined_data[f"{col_default}_{rep_name}"] = np.interp(
            reference_time, rep_data[time_col], rep_data[col])

# Step 3: Compute mean and std dev
for col_default in [temp_col_default, signal_col_default]:
    values = [combined_data[k] for k in combined_data if k.startswith(f"{col_default}_")]
    combined_data[f"{col_default}_Avg"] = np.mean(values, axis=0)
    combined_data[f"{col_default}_Std"] = np.std(values, axis=0)

# Step 4: Store and adjust columns
dsc_combined = pd.DataFrame(combined_data)

# Setp 5: Write data to file
file_label = f"DSC_{hr_label}_combined.csv"
file_path = os.path.join(output_dir, file_label)
dsc_combined.to_csv(file_path, index=False)


# Check result
dsc_combined.head()


In [ ]:
dsc_combined["Heat_Flow_Avg"]

In [ ]:

pmma_6mm = tupo_exp["6mm"]
hr_label = "5Kmin"


# Smoothing parameters
Δ_T = 10  # Kelvin
savgol_poly = 3





time = dsc_combined["Time"]
temp = dsc_combined["Temperature_Avg"]
dsc = dsc_combined["Heat_Flow_Avg"]

instantaneous_hr = np.gradient(temp, time)

# Smooth data
win_pts = utils.get_window_points(temp, Δ_T)
dsc_smooth = savgol_filter(dsc, window_length=win_pts, polyorder=savgol_poly)

cp = dsc_smooth / instantaneous_hr

start = 42  # cut off spike in the beginning
plt.plot(temp[start:] - 273.15, cp[start:])

# plt.plot(time,instantaneous_hr)

plt.plot([30,50,150,500], 
         [0.6,1.4,1.55,1.55], 
         color='black', marker='.')

        
# Plot meta data
plt.xlabel("Sample Temeprature / °C")
plt.ylabel("Normalised Heat Flow / J/(g K)")

# plt.xlim(right=500)
plt.xlim(right=300)
plt.ylim(bottom=0.5,top=2)

plt.grid()


In [ ]:
tupo_meta["6mm"]["10Kmin"]["Rep_01"]["SAMPLE MASS /mg"]

In [ ]:

pmma_6mm = tupo_exp["6mm"]
hr_label = "5Kmin"


# Smoothing parameters
Δ_T = 10  # Kelvin
savgol_poly = 3




time = dsc_combined["Time"] * 60
temp = dsc_combined["Temperature_Avg"]
dsc = dsc_combined["Heat_Flow_Avg"]

tga_meta = tupo_meta["6mm"][hr_label]
init_masses = list()
for rep_label in tga_meta:
    init_mass = tga_meta[rep_label]["SAMPLE MASS /mg"]
    init_masses.append(float(init_mass))
m_0 = np.average(init_masses) / 1000  # mg to g

const_hr = tupo_isoconv["experiments"]["TGA"]["constant_heating_rate"]
# m_t = m_0 * (const_hr[hr_label]["combined"]["Mass_Avg"] / 100)
m_t = const_hr[hr_label]["combined"]["Mass_Avg"] / 100

instantaneous_hr = np.gradient(temp, time)

# Smooth data
win_pts = utils.get_window_points(temp, Δ_T)
dsc_smooth = savgol_filter(dsc, window_length=win_pts, polyorder=savgol_poly)

norm_heat_flow = dsc_smooth / instantaneous_hr

T = [30,50,150,500]
cp_T = [0.6,1.4,1.55,1.55]

cp = np.interp(np.asarray(temp), T, cp_T)

baseline = utils.H_baseline(m_0=1, dT_dt=instantaneous_hr, c_T=cp/1000, m_t=m_t)


baseline = np.gradient(temp, time) * (cp * m_t)


start = 42  # cut off spike in the beginning
plt.plot(temp[start:], dsc_smooth[start:])


plt.plot(temp, baseline)


# Plot meta data
plt.xlabel("Sample Temeprature / °C")
plt.ylabel("Heat Flow / W/g")

plt.xlim(right=500)
# plt.xlim(right=300)
# plt.ylim(bottom=0.5,top=2)

plt.grid()
